# SQL Practice Notebook (Databricks)

In this notebook, we will cover:

- Creating tables
- Querying data
- Filtering, sorting
- Aggregations
- Joins
- Window functions
- CTEs and subqueries
- Writing data

All examples follow real-world data engineering scenarios.

## Step 1: Create Raw Table (Bronze Layer)

We simulate raw ingested data.

In [0]:
-- Create schema for real estate data
CREATE SCHEMA IF NOT EXISTS real_estate;

-- Use the schema
USE real_estate;

-- Create Bronze table for raw property transactions
CREATE TABLE IF NOT EXISTS bronze_property_transactions (
    transaction_id STRING COMMENT 'Unique transaction identifier',
    property_id STRING COMMENT 'Unique property identifier',
    buyer_id STRING COMMENT 'Unique buyer identifier',
    listing_price STRING COMMENT 'Original listing price (raw format)',
    sale_price STRING COMMENT 'Final sale price (raw, may contain errors)',
    property_type STRING COMMENT 'Type of property (e.g., apartment, house)',
    city STRING COMMENT 'City where property is located',
    listing_date STRING COMMENT 'Date when property was listed (raw format)',
    sale_date STRING COMMENT 'Date when property was sold (raw format)',
    agent_id STRING COMMENT 'Real estate agent identifier',
    ingestion_timestamp TIMESTAMP COMMENT 'Time when data was ingested'
)
USING DELTA
COMMENT 'Bronze layer table storing raw real estate transaction data';

Insert Raw Data (Simulating Ingestion)

In [0]:
INSERT INTO real_estate.bronze_property_transactions
VALUES
('T001', 'P001', 'B001', '500000', '480000', 'apartment', 'London', '2025-01-01', '2025-01-15', 'A001', current_timestamp()),
('T002', 'P002', 'B002', '750000', '740000', 'house', 'Manchester', '01-02-2025', '15-02-2025', 'A002', current_timestamp()),
('T003', 'P003', 'B003', '300000', 'unknown', 'apartment', 'Birmingham', '2025/03/01', '2025/03/20', 'A003', current_timestamp()),
('T004', 'P004', 'B004', '450000', '430000', 'house', 'Leeds', 'invalid_date', '2025-04-01', 'A004', current_timestamp());

In [0]:
-- -- Get the table structure
DESCRIBE TABLE real_estate.bronze_property_transactions;

## Step 2: Basic SELECT Queries

1️⃣ Select All Columns
- * selects every column.
- Use this when you want a quick overview of the data.

In [0]:
--  Select all rows and columns from the Bronze table
SELECT *
FROM real_estate.bronze_property_transactions;

2. Select Specific 
- Selecting only the columns you need can speed up queries.
- Useful for reporting or debugging specific fields.

In [0]:
-- Select only transaction ID, property ID, and sale price
SELECT transaction_id, property_id, sale_price
FROM real_estate.bronze_property_transactions;

💡 Explanation:

Selecting only the columns you need can speed up queries.
Useful for reporting or debugging specific fields.

3️⃣ Select Distinct Values
- DISTINCT removes duplicates.
- Great for understanding categorical columns in raw data.

In [0]:
-- Find all unique property types in the dataset
SELECT DISTINCT property_type
FROM real_estate.bronze_property_transactions;

4️⃣ Limit Rows (Preview Data)

- LIMIT is handy to quickly inspect the data without loading the entire table.

In [0]:
-- Preview only the first 5 rows
SELECT *
FROM real_estate.bronze_property_transactions
LIMIT 5;

5️⃣ Quick Aggregation Example
- Even in raw data, you can get useful metrics like total rows, unique buyers, etc.

In [0]:
-- Count total transactions in the Bronze table
SELECT COUNT(*) AS total_transactions
FROM real_estate.bronze_property_transactions;

## Step 3: Filtering Data

1️⃣ Filter by Exact Value
- Only rows where property_type equals 'apartment' are returned.

In [0]:
-- Show all apartment transactions
SELECT *
FROM real_estate.bronze_property_transactions
WHERE property_type = 'apartment';


2️⃣ Filter with Multiple Conditions
- AND combines multiple conditions.
- Use != or <> to exclude unwanted values.

In [0]:
-- Apartments sold in London with a known sale price
SELECT *
FROM real_estate.bronze_property_transactions
WHERE property_type = 'apartment'
  AND city = 'London'
  AND sale_price != 'unknown';

3️⃣ Filter Using Patterns
- % is a wildcard. 'L%' matches any city starting with "L" (e.g., London, Leeds).

In [0]:
-- Properties in cities that start with 'L'
SELECT *
FROM real_estate.bronze_property_transactions
WHERE city LIKE 'L%';

4️⃣ Filter with IN
- IN checks multiple values without repeating OR.

In [0]:
--- -- Show transactions in London or Manchester
SELECT *
FROM real_estate.bronze_property_transactions
WHERE city IN ('London', 'Manchester');

5️⃣ Filter with NULL or Missing Values
- Bronze data may have missing/invalid entries.
- Always check for NULL or placeholders like 'unknown'.

In [0]:

-- Transactions where sale_price is missing
SELECT *
FROM real_estate.bronze_property_transactions
WHERE sale_price IS NULL
   OR sale_price = 'unknown';

## Step 4: Sorting Results

1️⃣ Sort by One **Column**
- Default is ascending order.
- Useful for spotting low/high values.

In [0]:
-- Show transactions ordered by sale price (ascending)
SELECT *
FROM real_estate.bronze_property_transactions
ORDER BY sale_price;

2️⃣ Sort Descending

In [0]:
-- Show transactions ordered by sale price (highest first)
SELECT *
FROM real_estate.bronze_property_transactions
ORDER BY sale_price DESC;

3️⃣ Sort by Multiple Columns
- Rows are sorted primary by city, then within each city by sale_price.

In [0]:
-- Order first by city, then by sale_price descending
SELECT *
FROM real_estate.bronze_property_transactions
ORDER BY city, sale_price DESC;

4️⃣ Combine Filtering & Sorting
- Filtering reduces rows; sorting arranges them meaningfully.
- A common pattern for data exploration before transformations.

In [0]:
-- Apartments in London ordered by sale price descending
SELECT *
FROM real_estate.bronze_property_transactions
WHERE property_type = 'apartment'
  AND city = 'London'
ORDER BY sale_price DESC;

## Step 5: Aggregations

1️⃣ Count Total Transactions
- COUNT(*) counts all rows in the table.
- AS total_transactions gives a meaningful column name.

In [0]:
-- Count total property transactions
SELECT COUNT(*) AS total_transactions
FROM real_estate.bronze_property_transactions;

2️⃣ Count Transactions by City

- GROUP BY splits the table by city.
- COUNT(*) calculates the number of transactions in each group.****

In [0]:
-- Count transactions grouped by city
SELECT city, COUNT(*) AS transactions_per_city
FROM real_estate.bronze_property_transactions
GROUP BY city;

3️⃣ Sum & Average Sale Price
- Bronze layer may store numbers as strings → cast to DOUBLE.
- Exclude invalid entries using WHERE.
- Aggregation functions like SUM and AVG summarize data.

In [0]:
-- Sum and average of sale price by property type
SELECT property_type,
       SUM(CAST(sale_price AS DOUBLE)) AS total_sales,
       AVG(CAST(sale_price AS DOUBLE)) AS avg_sale_price
FROM real_estate.bronze_property_transactions
WHERE sale_price != 'unknown'
GROUP BY property_type;

4️⃣ Max & Min Sale Price by City
- MAX and MIN find extreme values in each group.

In [0]:
-- Highest and lowest sale price per city
SELECT city,
       MAX(CAST(sale_price AS DOUBLE)) AS max_price,
       MIN(CAST(sale_price AS DOUBLE)) AS min_price
FROM real_estate.bronze_property_transactions
WHERE sale_price != 'unknown'
GROUP BY city;


%md
## Step 6: HAVING Clause

1 Using HAVING to Filter Groups
- HAVING filters the aggregated results.
- WHERE cannot filter aggregated results; that’s the difference.

In [0]:
-- Cities with more than 2 transactions
SELECT city,
       COUNT(*) AS transaction_count
FROM real_estate.bronze_property_transactions
GROUP BY city
HAVING COUNT(*) > 2;

2 Multiple Aggregations with HAVING

- Filters groups by a calculated metric (average price).
- Very common in reporting and dashboards.

In [0]:

-- Property types with average sale price over 500,000
SELECT property_type,
       AVG(CAST(sale_price AS DOUBLE)) AS avg_price,
       COUNT(*) AS transaction_count
FROM real_estate.bronze_property_transactions
WHERE sale_price != 'unknown'
GROUP BY property_type
HAVING AVG(CAST(sale_price AS DOUBLE)) > 500000;


## Step 7: Create Customer Table (Dimension)

Primary Key:
customer_id uniquely identifies each customer.
This links to buyer_id in your bronze_property_transactions table.
Descriptive Attributes:
Names, email, phone, location, registration date, and VIP status are typical dimension fields.
Ingestion Timestamp:
Tracks when the dimension record was loaded/updated.
Useful for incremental updates or auditing.
Delta Table:
Supports ACID transactions and time travel.
Perfect for slowly changing dimensions or incremental updates.

In [0]:
-- Create a Bronze/Dimension schema if not already
CREATE SCHEMA IF NOT EXISTS real_estate;

-- Use the schema
USE real_estate;

-- Create Customer dimension table
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_id STRING COMMENT 'Unique customer identifier, matches buyer_id in transactions',
    first_name STRING COMMENT 'Customer first name',
    last_name STRING COMMENT 'Customer last name',
    email STRING COMMENT 'Customer email address',
    phone STRING COMMENT 'Customer phone number',
    city STRING COMMENT 'City where the customer lives',
    state STRING COMMENT 'State or province',
    country STRING COMMENT 'Country',
    registration_date DATE COMMENT 'Date the customer registered in the system',
    vip_status STRING COMMENT 'VIP status: e.g., Gold, Silver, Bronze',
    ingestion_timestamp TIMESTAMP COMMENT 'Timestamp when this record was ingested'
)
USING DELTA
COMMENT 'Customer dimension table storing descriptive attributes for buyers';

In [0]:
INSERT INTO dim_customer VALUES
('B001', 'Alice', 'Johnson', 'alice.johnson@example.com', '555-1234', 'London', 'England', 'UK', '2023-01-15', 'Gold', current_timestamp()),
('B002', 'Bob', 'Smith', 'bob.smith@example.com', '555-5678', 'Manchester', 'England', 'UK', '2023-03-22', 'Silver', current_timestamp()),
('B003', 'Charlie', 'Brown', 'charlie.brown@example.com', '555-9876', 'Birmingham', 'England', 'UK', '2023-05-10', 'Bronze', current_timestamp());

## Step 8: Joins

1️⃣ INNER JOIN

INNER JOIN returns rows where the buyer_id exists in both tables.
Transactions with no matching customer are excluded.

In [0]:
-- Get all transactions along with customer info (only matches)
SELECT 
    t.transaction_id,
    t.sale_price,
    t.property_type,
    c.first_name,
    c.last_name,
    c.city AS customer_city,
    c.vip_status
FROM real_estate.bronze_property_transactions t
INNER JOIN real_estate.dim_customer c
ON t.buyer_id = c.customer_id;

2️⃣ LEFT JOIN (Most Common in Analytics)

LEFT JOIN keeps all transactions, even if the customer record is missing.
Useful for raw Bronze data where dimensions may be incomplete.

In [0]:
-- Get all transactions, include customer info if available
SELECT 
    t.transaction_id,
    t.sale_price,
    t.property_type,
    c.first_name,
    c.last_name,
    c.city AS customer_city
FROM real_estate.bronze_property_transactions t
LEFT JOIN real_estate.dim_customer c
ON t.buyer_id = c.customer_id;

3️⃣ RIGHT JOIN

RIGHT JOIN keeps all customers, even if they have no transactions.
Less common, but useful for customer reporting.

In [0]:

-- Get all customers, include transactions if available
SELECT 
    t.transaction_id,
    t.sale_price,
    c.first_name,
    c.last_name
FROM real_estate.bronze_property_transactions t
RIGHT JOIN real_estate.dim_customer c
ON t.buyer_id = c.customer_id;

4️⃣ FULL OUTER JOIN

Returns all rows from both tables.
Missing matches appear as NULL.
Good for data quality checks.

In [0]:
-- Include all transactions and all customers
SELECT 
    t.transaction_id,
    t.sale_price,
    c.first_name,
    c.last_name
FROM real_estate.bronze_property_transactions t
FULL OUTER JOIN real_estate.dim_customer c
ON t.buyer_id = c.customer_id;


5️⃣ JOIN with Filtering & Aggregation

Combines filtering, aggregation, and join.
Shows average sale price by VIP customer type.

In [0]:
-- Count transactions per VIP status
SELECT 
    c.vip_status,
    COUNT(t.transaction_id) AS transactions_count,
    AVG(CAST(t.sale_price AS DOUBLE)) AS avg_sale_price
FROM real_estate.bronze_property_transactions t
LEFT JOIN real_estate.dim_customer c
ON t.buyer_id = c.customer_id
WHERE t.sale_price != 'unknown'
GROUP BY c.vip_status
ORDER BY avg_sale_price DESC;


## Step 9: Conditional Logic

1️⃣ Using CASE WHEN

CASE WHEN works like if-else in programming.
Useful for categorizing, bucketing, or flagging data.
Always handle invalid/missing values first (e.g., 'unknown').

In [0]:
-- Categorize transactions based on sale price
SELECT 
    transaction_id,
    sale_price,
    property_type,
    CASE 
        WHEN sale_price = 'unknown' THEN 'Unknown Price'
        WHEN CAST(sale_price AS DOUBLE) < 500000 THEN 'Low Price'
        WHEN CAST(sale_price AS DOUBLE) BETWEEN 500000 AND 800000 THEN 'Medium Price'
        ELSE 'High Price'
    END AS price_category
FROM real_estate.bronze_property_transactions;

2️⃣ Conditional Logic with JOINs

Combines conditional logic with dimension join.
Allows you to flag important records for analysis.

In [0]:
-- Flag VIP customers and their transaction size
SELECT 
    t.transaction_id,
    t.sale_price,
    c.vip_status,
    CASE 
        WHEN c.vip_status = 'Gold' AND CAST(t.sale_price AS DOUBLE) > 700000 THEN 'High Value Gold'
        WHEN c.vip_status = 'Silver' AND CAST(t.sale_price AS DOUBLE) > 500000 THEN 'High Value Silver'
        ELSE 'Standard'
    END AS vip_transaction_flag
FROM real_estate.bronze_property_transactions t
LEFT JOIN real_estate.dim_customer c
ON t.buyer_id = c.customer_id;


3️⃣ Using IF() (Databricks SQL supports IF)

IF(condition, value_if_true, value_if_false) is shorter than CASE WHEN for simple conditions.

In [0]:
-- Simple conditional: if sale_price unknown, mark as missing
SELECT 
    transaction_id,
    sale_price,
    IF(sale_price = 'unknown', 'Missing', 'Known') AS price_status
FROM real_estate.bronze_property_transactions;

4️⃣ Multiple Conditions Example


You can combine multiple columns in a single conditional logic block.
Great for data labeling and analytics-ready transformations.

In [0]:
-- Categorize properties based on type and sale price safely
SELECT 
    transaction_id,
    property_type,
    sale_price,
    CASE
        WHEN property_type = 'apartment' AND TRY_CAST(sale_price AS DOUBLE) < 500000 THEN 'Cheap Apartment'
        WHEN property_type = 'house' AND TRY_CAST(sale_price AS DOUBLE) >= 500000 THEN 'Expensive House'
        ELSE 'Other'
    END AS property_label
FROM real_estate.bronze_property_transactions;

## Step 10: Date Functions

In [0]:
-- Safely convert listing_date and sale_date to DATE/TIMESTAMP
SELECT
    transaction_id,
    listing_date,
    sale_date,
    TRY_TO_DATE(listing_date, 'yyyy-MM-dd') AS listing_date_parsed,
    TRY_TO_DATE(sale_date, 'yyyy-MM-dd') AS sale_date_parsed,
    TRY_TO_TIMESTAMP(listing_date, 'yyyy-MM-dd') AS listing_timestamp,
    TRY_TO_TIMESTAMP(sale_date, 'yyyy-MM-dd') AS sale_timestamp
FROM real_estate.bronze_property_transactions;

In [0]:
-- Safely convert listing_date and sale_date to DATE/TIMESTAMP
SELECT
    transaction_id,
    listing_date,
    sale_date,
    TRY_TO_DATE(listing_date, 'yyyy-MM-dd') AS listing_date_parsed,
    TRY_TO_DATE(sale_date, 'yyyy-MM-dd') AS sale_date_parsed,
    TRY_TO_TIMESTAMP(listing_date, 'yyyy-MM-dd') AS listing_timestamp,
    TRY_TO_TIMESTAMP(sale_date, 'yyyy-MM-dd') AS sale_timestamp
FROM real_estate.bronze_property_transactions;

In [0]:
-- Extract year, month, and day from listing_date safely
SELECT
    transaction_id,
    listing_date,
    YEAR(TRY_TO_DATE(listing_date, 'yyyy-MM-dd')) AS listing_year,
    MONTH(TRY_TO_DATE(listing_date, 'yyyy-MM-dd')) AS listing_month,
    DAY(TRY_TO_DATE(listing_date, 'yyyy-MM-dd')) AS listing_day
FROM real_estate.bronze_property_transactions;

In [0]:
-- Calculate days to sell safely
SELECT
    transaction_id,
    listing_date,
    sale_date,
    DATEDIFF(
        TRY_TO_DATE(sale_date, 'yyyy-MM-dd'),
        TRY_TO_DATE(listing_date, 'yyyy-MM-dd')
    ) AS days_to_sell
FROM real_estate.bronze_property_transactions;

In [0]:
-- Add current date and calculate days since sale
SELECT
    transaction_id,
    sale_date,
    CURRENT_DATE() AS today_date,
    CURRENT_TIMESTAMP() AS current_timestamp,
    DATEDIFF(CURRENT_DATE(), TRY_TO_DATE(sale_date, 'yyyy-MM-dd')) AS days_since_sale
FROM real_estate.bronze_property_transactions;

In [0]:
-- Safely calculate future/past dates
SELECT
    transaction_id,
    sale_date,
    DATE_ADD(TRY_TO_DATE(sale_date, 'yyyy-MM-dd'), 180) AS next_inspection_date,
    DATE_SUB(TRY_TO_DATE(sale_date, 'yyyy-MM-dd'), 30) AS thirty_days_before_sale,
    ADD_MONTHS(TRY_TO_DATE(sale_date, 'yyyy-MM-dd'), 6) AS six_months_later
FROM real_estate.bronze_property_transactions;

In [0]:
-- Format dates safely for reports
SELECT
    transaction_id,
    sale_date,
    DATE_FORMAT(TRY_TO_DATE(sale_date, 'yyyy-MM-dd'), 'dd-MMM-yyyy') AS sale_date_formatted
FROM real_estate.bronze_property_transactions;

In [0]:
-- Try multiple formats using COALESCE
SELECT
    transaction_id,
    COALESCE(
        TRY_TO_DATE(listing_date, 'yyyy-MM-dd'),
        TRY_TO_DATE(listing_date, 'dd-MM-yyyy'),
        TRY_TO_DATE(listing_date, 'yyyy/MM/dd')
    ) AS listing_date_clean,
    COALESCE(
        TRY_TO_DATE(sale_date, 'yyyy-MM-dd'),
        TRY_TO_DATE(sale_date, 'dd-MM-yyyy'),
        TRY_TO_DATE(sale_date, 'yyyy/MM/dd')
    ) AS sale_date_clean
FROM real_estate.bronze_property_transactions;

## Step 11: Window Functions

In [0]:
SELECT
    transaction_id,
    city,
    property_type,
    TRY_CAST(sale_price AS DOUBLE) AS sale_price_numeric,
    ROW_NUMBER() OVER (
        PARTITION BY city
        ORDER BY TRY_CAST(sale_price AS DOUBLE) DESC
    ) AS rank_in_city
FROM real_estate.bronze_property_transactions;

In [0]:
SELECT
    transaction_id,
    city,
    TRY_CAST(sale_price AS DOUBLE) AS sale_price_numeric,
    RANK() OVER (
        PARTITION BY city
        ORDER BY TRY_CAST(sale_price AS DOUBLE) DESC
    ) AS rank_in_city
FROM real_estate.bronze_property_transactions;

In [0]:
SELECT
    transaction_id,
    city,
    TRY_CAST(sale_price AS DOUBLE) AS sale_price_numeric,
    SUM(TRY_CAST(sale_price AS DOUBLE)) OVER (
        PARTITION BY city
        ORDER BY TRY_TO_DATE(sale_date, 'yyyy-MM-dd')
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_sales
FROM real_estate.bronze_property_transactions;

In [0]:
SELECT
    transaction_id,
    city,
    TRY_CAST(sale_price AS DOUBLE) AS sale_price_numeric,
    LAG(TRY_CAST(sale_price AS DOUBLE), 1) OVER (
        PARTITION BY city
        ORDER BY TRY_TO_DATE(sale_date, 'yyyy-MM-dd')
    ) AS prev_sale_price,
    LEAD(TRY_CAST(sale_price AS DOUBLE), 1) OVER (
        PARTITION BY city
        ORDER BY TRY_TO_DATE(sale_date, 'yyyy-MM-dd')
    ) AS next_sale_price
FROM real_estate.bronze_property_transactions;

## Step 12: CTE

In [0]:
-- Use CTE to select high-value transactions first
WITH high_value_transactions AS (
    SELECT *
    FROM real_estate.bronze_property_transactions
    WHERE TRY_CAST(sale_price AS DOUBLE) > 700000
)
SELECT *
FROM high_value_transactions;

In [0]:
-- Rank transactions per city and select top 2 per city
WITH ranked_transactions AS (
    SELECT
        transaction_id,
        city,
        property_type,
        TRY_CAST(sale_price AS DOUBLE) AS sale_price_numeric,
        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY TRY_CAST(sale_price AS DOUBLE) DESC
        ) AS rank_in_city
    FROM real_estate.bronze_property_transactions
)
SELECT *
FROM ranked_transactions
WHERE rank_in_city <= 2;

In [0]:
-- Average sale price per city using CTE
WITH city_sales AS (
    SELECT
        city,
        AVG(TRY_CAST(sale_price AS DOUBLE)) AS avg_sale_price,
        COUNT(*) AS transactions_count
    FROM real_estate.bronze_property_transactions
    WHERE sale_price != 'unknown'
    GROUP BY city
)
SELECT *
FROM city_sales
WHERE avg_sale_price > 500000
ORDER BY avg_sale_price DESC;

## Step 13: Subqueries

In [0]:
-- Select transactions with sale price above average
SELECT transaction_id, sale_price, property_type
FROM real_estate.bronze_property_transactions
WHERE TRY_CAST(sale_price AS DOUBLE) >
      (SELECT AVG(TRY_CAST(sale_price AS DOUBLE))
       FROM real_estate.bronze_property_transactions
       WHERE sale_price != 'unknown');

In [0]:
-- Calculate total sales per city, then select top cities
SELECT city, total_sales
FROM (
    SELECT city, SUM(TRY_CAST(sale_price AS DOUBLE)) AS total_sales
    FROM real_estate.bronze_property_transactions
    WHERE sale_price != 'unknown'
    GROUP BY city
) AS city_totals
WHERE total_sales > 1000000
ORDER BY total_sales DESC;

In [0]:
-- Select transactions with highest sale price per city
SELECT t.transaction_id, t.city, t.sale_price
FROM real_estate.bronze_property_transactions t
WHERE TRY_CAST(sale_price AS DOUBLE) = (
    SELECT MAX(TRY_CAST(sale_price AS DOUBLE))
    FROM real_estate.bronze_property_transactions
    WHERE city = t.city
);

In [0]:
-- Get transactions where customer is VIP Gold
SELECT t.transaction_id, t.sale_price, c.first_name, c.vip_status
FROM real_estate.bronze_property_transactions t
JOIN real_estate.dim_customer c
ON t.buyer_id = c.customer_id
WHERE c.customer_id IN (
    SELECT customer_id
    FROM real_estate.dim_customer
    WHERE vip_status = 'Gold'
);

In [0]:
-- Flag transactions above city average
SELECT t.transaction_id, t.city, t.sale_price,
       CASE
           WHEN TRY_CAST(t.sale_price AS DOUBLE) >
                (SELECT AVG(TRY_CAST(sale_price AS DOUBLE))
                 FROM real_estate.bronze_property_transactions
                 WHERE city = t.city AND sale_price != 'unknown')
           THEN 'Above Avg'
           ELSE 'Below Avg'
       END AS sale_category
FROM real_estate.bronze_property_transactions t;

## Step 14: Create Cleaned Table (Silver Layer)

In [0]:
-- Create Silver table for cleaned property transactions
CREATE TABLE IF NOT EXISTS real_estate.silver_property_transactions (
    transaction_id STRING COMMENT 'Unique transaction identifier',
    property_id STRING COMMENT 'Unique property identifier',
    buyer_id STRING COMMENT 'Unique buyer identifier',
    listing_price DOUBLE COMMENT 'Original listing price (numeric, cleaned)',
    sale_price DOUBLE COMMENT 'Final sale price (numeric, cleaned)',
    property_type STRING COMMENT 'Type of property (apartment, house, etc.)',
    city STRING COMMENT 'City where property is located',
    listing_date DATE COMMENT 'Cleaned listing date',
    sale_date DATE COMMENT 'Cleaned sale date',
    agent_id STRING COMMENT 'Real estate agent identifier',
    vip_status STRING COMMENT 'Customer VIP status from dimension table',
    price_category STRING COMMENT 'Categorical flag based on sale price',
    days_to_sell INT COMMENT 'Days between listing and sale',
    ingestion_timestamp TIMESTAMP COMMENT 'Time when data was ingested'
)
USING DELTA
COMMENT 'Silver layer table: cleaned and transformed property transactions';

In [0]:
INSERT INTO real_estate.silver_property_transactions
SELECT
    t.transaction_id,
    t.property_id,
    t.buyer_id,
    TRY_CAST(t.listing_price AS DOUBLE) AS listing_price,
    TRY_CAST(t.sale_price AS DOUBLE) AS sale_price,
    t.property_type,
    t.city,
    TRY_TO_DATE(t.listing_date, 'yyyy-MM-dd') AS listing_date,
    TRY_TO_DATE(t.sale_date, 'yyyy-MM-dd') AS sale_date,
    t.agent_id,
    c.vip_status,
    CASE
        WHEN TRY_CAST(t.sale_price AS DOUBLE) IS NULL THEN 'Unknown'
        WHEN TRY_CAST(t.sale_price AS DOUBLE) < 500000 THEN 'Low Price'
        WHEN TRY_CAST(t.sale_price AS DOUBLE) BETWEEN 500000 AND 800000 THEN 'Medium Price'
        ELSE 'High Price'
    END AS price_category,
    DATEDIFF(TRY_TO_DATE(t.sale_date, 'yyyy-MM-dd'),
             TRY_TO_DATE(t.listing_date, 'yyyy-MM-dd')) AS days_to_sell,
    t.ingestion_timestamp
FROM real_estate.bronze_property_transactions t
LEFT JOIN real_estate.dim_customer c
ON t.buyer_id = c.customer_id;

In [0]:
-- Check data after transformation
SELECT *
FROM real_estate.silver_property_transactions
LIMIT 20;

## Step 15: Create Aggregated Table (Gold Layer)


In [0]:
-- Create Gold table for aggregated property sales
CREATE TABLE IF NOT EXISTS real_estate.gold_property_sales_summary (
    city STRING COMMENT 'City where property is located',
    property_type STRING COMMENT 'Type of property',
    total_transactions INT COMMENT 'Total number of transactions',
    total_sales DOUBLE COMMENT 'Sum of sale prices',
    avg_sale_price DOUBLE COMMENT 'Average sale price',
    max_sale_price DOUBLE COMMENT 'Maximum sale price',
    min_sale_price DOUBLE COMMENT 'Minimum sale price',
    avg_days_to_sell DOUBLE COMMENT 'Average days between listing and sale',
    high_value_transactions INT COMMENT 'Number of transactions above 800k',
    vip_transactions INT COMMENT 'Number of transactions by VIP customers',
    ingestion_timestamp TIMESTAMP COMMENT 'Time when this aggregation was created'
)
USING DELTA
COMMENT 'Gold layer table: aggregated metrics for real estate sales';

In [0]:
INSERT INTO real_estate.gold_property_sales_summary
SELECT
    city,
    property_type,
    COUNT(transaction_id) AS total_transactions,
    SUM(sale_price) AS total_sales,
    AVG(sale_price) AS avg_sale_price,
    MAX(sale_price) AS max_sale_price,
    MIN(sale_price) AS min_sale_price,
    AVG(days_to_sell) AS avg_days_to_sell,
    SUM(CASE WHEN sale_price >= 800000 THEN 1 ELSE 0 END) AS high_value_transactions,
    SUM(CASE WHEN vip_status = 'Gold' THEN 1 ELSE 0 END) AS vip_transactions,
    CURRENT_TIMESTAMP() AS ingestion_timestamp
FROM real_estate.silver_property_transactions
WHERE sale_price IS NOT NULL
GROUP BY city, property_type
ORDER BY city, property_type;

In [0]:
-- Example: Top 5 cities with highest average sale price
SELECT city, property_type, avg_sale_price, total_transactions
FROM real_estate.gold_property_sales_summary
ORDER BY avg_sale_price DESC
LIMIT 5;

## Step 16: Query Gold Table

In [0]:
SELECT *
FROM real_estate.gold_property_sales_summary
ORDER BY city, property_type;

In [0]:
SELECT city, property_type, avg_sale_price, total_transactions
FROM real_estate.gold_property_sales_summary
ORDER BY avg_sale_price DESC
LIMIT 5;

In [0]:
SELECT city, property_type, high_value_transactions
FROM real_estate.gold_property_sales_summary
ORDER BY high_value_transactions DESC
LIMIT 10;

In [0]:
SELECT city, property_type, vip_transactions, total_transactions,
       ROUND((vip_transactions / total_transactions) * 100, 2) AS vip_percentage
FROM real_estate.gold_property_sales_summary
ORDER BY vip_percentage DESC
LIMIT 10;

In [0]:
SELECT *
FROM real_estate.gold_property_sales_summary
WHERE avg_sale_price >= 700000 AND total_transactions >= 50
ORDER BY avg_sale_price DESC;

In [0]:
-- Join back to Silver to see individual transactions in top-performing cities
SELECT s.transaction_id, s.sale_price, s.property_type, s.city, g.avg_sale_price
FROM real_estate.silver_property_transactions s
JOIN real_estate.gold_property_sales_summary g
  ON s.city = g.city AND s.property_type = g.property_type
WHERE g.avg_sale_price >= 700000;

## Step 17: Create View

In [0]:
-- Create a view on top of the Gold table
CREATE OR REPLACE VIEW real_estate.vw_gold_property_sales_summary AS
SELECT *
FROM real_estate.gold_property_sales_summary;

In [0]:
SELECT city, property_type, avg_sale_price, total_transactions
FROM real_estate.vw_gold_property_sales_summary
WHERE high_value_transactions > 10
ORDER BY avg_sale_price DESC;

## Step 18: Optimize Table (Delta Lake)

In [0]:
-- Optimize the Gold table for faster queries
OPTIMIZE real_estate.gold_property_sales_summary;

In [0]:
-- Optimize with Z-ORDER for frequently filtered columns
OPTIMIZE real_estate.gold_property_sales_summary
ZORDER BY (city, property_type);

## Key Learnings

- SQL is the backbone of data engineering
- Window functions are heavily used in interviews
- Always think in layers:
    - Bronze → Raw
    - Silver → Cleaned
    - Gold → Aggregated
- Use CTEs for readability
- Optimize tables for performance in Databricks